In [ ]:
# %% [markdown]
# # Analyze Training Log
# 
# This notebook parses a RetinaFace training log file (e.g., `retinaface_train_14_04_2025.log`)
# and extracts the metrics from every valid line (lines containing `"Epoch:"`).
# The metrics extracted are:
# 
# - Epoch and Batch information
# - **Loss** (current and average)
# - **Loc** (current and average)
# - **Cls** (current and average)
# 
# Then, it creates an interactive plot using Plotly to help visualize how these values evolve over the training process.

# %% [code]
import re
import pandas as pd
import plotly.graph_objects as go

# Path to your log file
logfile_path = "retinaface_train_14_04_2025.log"

# Read in the log file
with open(logfile_path, "r") as file:
    lines = file.readlines()

# %% [markdown]
# ## Parsing the Log
# 
# We assume that each valid log line has a structure similar to:
# 
# ```
# Epoch: [2/80][210/460] ETA: 6:56:27 LR: 0.00480543 Time: 0.6935 (0.7005) Data: 0.0209 (0.0263) Loss: 1.3008 (1.7318) Loc: 0.9688 (1.2383) Cls: 0.3320 (0.4935)
# ```
# 
# Here we use a regular expression to extract:
# 
# - Current epoch and total epochs.
# - Current batch (iteration) and total batches.
# - For each of the metrics: the "current" and the "average" (in parentheses) values.
# 
# We then build a DataFrame out of these values.

# %% [code]
# Define a regex pattern for matching the log lines with required metrics.
pattern = re.compile(
    r"Epoch:\s+\[(\d+)/(\d+)\]\[(\d+)/(\d+)\].*Loss:\s+([0-9.]+)\s+\(([0-9.]+)\).*Loc:\s+([0-9.]+)\s+\(([0-9.]+)\).*Cls:\s+([0-9.]+)\s+\(([0-9.]+)\)"
)

data = []
global_step = 0  # A counter to track global iteration across epochs

for line in lines:
    if "Epoch:" in line:
        match = pattern.search(line)
        if match:
            epoch = int(match.group(1))
            total_epochs = int(match.group(2))
            batch = int(match.group(3))
            total_batches = int(match.group(4))
            loss_cur = float(match.group(5))
            loss_avg = float(match.group(6))
            loc_cur = float(match.group(7))
            loc_avg = float(match.group(8))
            cls_cur = float(match.group(9))
            cls_avg = float(match.group(10))
            
            # Append the extracted values as a dictionary
            data.append({
                "epoch": epoch,
                "total_epochs": total_epochs,
                "batch": batch,
                "total_batches": total_batches,
                "global_step": global_step,
                "loss_cur": loss_cur,
                "loss_avg": loss_avg,
                "loc_cur": loc_cur,
                "loc_avg": loc_avg,
                "cls_cur": cls_cur,
                "cls_avg": cls_avg
            })
            global_step += 1

# Convert list of dictionaries into a pandas DataFrame.
df = pd.DataFrame(data)

# Display the first few rows to verify that the data has been parsed correctly.
df.head()

# %% [markdown]
# ## Interactive Visualization with Plotly
# 
# In the next cell, we create an interactive figure using Plotly. Here are some notes:
# 
# - The **x-axis** uses the `global_step` (i.e. each valid log line in sequence).
# - We add one trace per metric. In this example, both the "current" and "average" values for each of `Loss`, `Loc`, and `Cls` are added.
# - Plotly's interactive features (zooming, panning, hovering) will let you explore the trends.
# 
# If you want a more refined UI (for example, adding dropdown menus to filter which metric to see), you can further expand on this.

# %% [code]
# Build the interactive plot
fig = go.Figure()

# Add traces for Loss (current & average)
fig.add_trace(go.Scatter(x=df["global_step"], y=df["loss_cur"], mode='lines+markers', name="Loss (Current)"))
fig.add_trace(go.Scatter(x=df["global_step"], y=df["loss_avg"], mode='lines+markers', name="Loss (Average)"))

# Add traces for Loc (current & average)
fig.add_trace(go.Scatter(x=df["global_step"], y=df["loc_cur"], mode='lines+markers', name="Loc (Current)"))
fig.add_trace(go.Scatter(x=df["global_step"], y=df["loc_avg"], mode='lines+markers', name="Loc (Average)"))

# Add traces for Cls (current & average)
fig.add_trace(go.Scatter(x=df["global_step"], y=df["cls_cur"], mode='lines+markers', name="Cls (Current)"))
fig.add_trace(go.Scatter(x=df["global_step"], y=df["cls_avg"], mode='lines+markers', name="Cls (Average)"))

fig.update_layout(
    title="Training Metrics Over Global Steps",
    xaxis_title="Global Step (Iteration)",
    yaxis_title="Metric Value",
    hovermode="closest"
)

fig.show()

# %% [markdown]
# ### Extra Notes
# 
# - **Running the Notebook:** Open this notebook in Jupyter Notebook or JupyterLab to interact with the Plotly figure.
# - **File Path:** Make sure that the `logfile_path` variable is correctly set to the location of your log file.
# - **Further Interactivity:** You can expand the code with dropdowns or sliders using Plotly's built-in UI components or even with `ipywidgets` if you need more control.
